# 6.1 Metadata Trial 002: PyCaret Benchmark

Purpose:
- Benchmark classical ML models on metadata features.
- Compare PyCaret-selected model against Trial 001.
- Evaluate final model on held-out test split.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from pycaret.classification import (
    setup,
    compare_models,
    pull,
    predict_model,
    save_model,
)

from rural_stroke_assist.config import (
    METADATA_SPLIT_MANIFEST_PATH,
    METADATA_TRIAL_002_MODEL_PATH,
    METADATA_TRIAL_002_RESULTS_PATH,
    METADATA_TRIAL_002_LEADERBOARD_PATH,
    METADATA_TRIAL_002_PREDICTIONS_PATH,
    METADATA_TRIAL_002_CONFUSION_MATRIX_PATH,
)

from rural_stroke_assist.modeling.pycaret_metadata import (
    split_metadata_pycaret_data,
    prepare_pycaret_metadata_dataframe,
    build_pycaret_metadata_train_data,
)

from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
    save_confusion_matrix_plot,
    save_markdown_results,
)

In [2]:
metadata_df = pd.read_csv(METADATA_SPLIT_MANIFEST_PATH)

metadata_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke,split
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1,train
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1,train
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1,train
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1,train
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1,train


In [3]:
metadata_df.groupby(["split", "stroke"]).size()

split  stroke
test   0          729
       1           38
train  0         3403
       1          174
val    0          729
       1           37
dtype: int64

In [4]:
train_df, val_df, test_df = split_metadata_pycaret_data(metadata_df)

pycaret_train_source_df = build_pycaret_metadata_train_data(train_df, val_df)

pycaret_train_df = prepare_pycaret_metadata_dataframe(pycaret_train_source_df)
pycaret_test_df = prepare_pycaret_metadata_dataframe(test_df)

pycaret_train_df.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,gender,ever_married,work_type,Residence_type,smoking_status,stroke
0,67.0,0,1,228.69,36.6,Male,Yes,Private,Urban,formerly smoked,1
1,61.0,0,0,202.21,NaN,Female,Yes,Self-employed,Rural,never smoked,1
2,80.0,0,1,105.92,32.5,Male,Yes,Private,Rural,never smoked,1
3,49.0,0,0,171.23,34.4,Female,Yes,Private,Urban,smokes,1
4,79.0,1,0,174.12,24.0,Female,Yes,Self-employed,Rural,never smoked,1


In [5]:
experiment = setup(
    data=pycaret_train_df,
    target="stroke",
    session_id=42,
    normalize=True,
    fix_imbalance=True,
    fold=5,
    html=False,
    verbose=True,
)

                    Description             Value
0                    Session id                42
1                        Target            stroke
2                   Target type            Binary
3           Original data shape        (4343, 11)
4        Transformed data shape        (7087, 20)
5   Transformed train set shape        (5784, 20)
6    Transformed test set shape        (1303, 20)
7              Numeric features                 5
8          Categorical features                 5
9      Rows with missing values              4.1%
10                   Preprocess              True
11              Imputation type            simple
12           Numeric imputation              mean
13       Categorical imputation              mode
14     Maximum one-hot encoding                25
15              Encoding method              None
16                Fix imbalance              True
17         Fix imbalance method             SMOTE
18                    Normalize              True


In [6]:
best_models = compare_models(
    sort="AUC",
    n_select=5,
)

                                    Model  Accuracy     AUC  Recall   Prec.  \
ridge                    Ridge Classifier    0.7345  0.8338  0.7910  0.1310   
lda          Linear Discriminant Analysis    0.7345  0.8338  0.7910  0.1310   
lr                    Logistic Regression    0.7464  0.8306  0.7841  0.1357   
svm                   SVM - Linear Kernel    0.7224  0.8189  0.7703  0.1234   
lightgbm  Light Gradient Boosting Machine    0.9414  0.8113  0.0476  0.1475   
gbc          Gradient Boosting Classifier    0.9480  0.8099  0.0202  0.2167   
ada                  Ada Boost Classifier    0.9408  0.8044  0.0605  0.1650   
nb                            Naive Bayes    0.2069  0.7983  0.9867  0.0572   
rf               Random Forest Classifier    0.9464  0.7830  0.0340  0.1967   
et                 Extra Trees Classifier    0.9405  0.7599  0.0543  0.1795   
knn                K Neighbors Classifier    0.8549  0.6559  0.2646  0.1054   
qda       Quadratic Discriminant Analysis    0.7589 

In [7]:
leaderboard_df = pull()

METADATA_TRIAL_002_LEADERBOARD_PATH.parent.mkdir(parents=True, exist_ok=True)
leaderboard_df.to_csv(METADATA_TRIAL_002_LEADERBOARD_PATH, index=False)

leaderboard_df

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
ridge,Ridge Classifier,0.7345,0.8338,0.7910,0.1310,0.2247,0.1541,0.2469,0.034
lda,Linear Discriminant Analysis,0.7345,0.8338,0.7910,0.1310,0.2247,0.1541,0.2469,0.028
lr,Logistic Regression,0.7464,0.8306,0.7841,0.1357,0.2312,0.1616,0.2530,0.884
svm,SVM - Linear Kernel,0.7224,0.8189,0.7703,0.1234,0.2126,0.1405,0.2294,0.032
lightgbm,Light Gradient Boosting Machine,0.9414,0.8113,0.0476,0.1475,0.0718,0.0509,0.0592,0.204
gbc,Gradient Boosting Classifier,0.9480,0.8099,0.0202,0.2167,0.0368,0.0277,0.0516,0.234
ada,Ada Boost Classifier,0.9408,0.8044,0.0605,0.1650,0.0871,0.0649,0.0736,0.086
nb,Naive Bayes,0.2069,0.7983,0.9867,0.0572,0.1081,0.0177,0.0899,0.592
rf,Random Forest Classifier,0.9464,0.7830,0.0340,0.1967,0.0579,0.0445,0.0635,0.098
et,Extra Trees Classifier,0.9405,0.7599,0.0543,0.1795,0.0821,0.0597,0.0723,0.084


In [8]:
best_model = best_models[0] if isinstance(best_models, list) else best_models

best_model

RidgeClassifier(alpha=1.0, class_weight=None, copy_X=True, fit_intercept=True,
                max_iter=None, positive=False, random_state=42, solver='auto',
                tol=0.0001)

In [9]:
pycaret_predictions = predict_model(
    best_model,
    data=pycaret_test_df,
)

pycaret_predictions.head()

              Model  Accuracy     AUC  Recall  Prec.      F1   Kappa     MCC
0  Ridge Classifier    0.7053  0.7577  0.8158  0.124  0.2153  0.1414  0.2386


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender,ever_married,work_type,Residence_type,smoking_status,stroke,prediction_label
5,81.0,0,0,186.210007,29.000000,Male,Yes,Private,Urban,formerly smoked,1,1
24,71.0,0,0,102.870003,27.200001,Male,Yes,Private,Urban,formerly smoked,1,1
31,42.0,0,0,83.410004,25.400000,Male,Yes,Private,Rural,Unknown,1,0
51,78.0,1,0,75.320000,NaN,Male,Yes,Private,Urban,formerly smoked,1,1
53,77.0,1,0,124.129997,31.400000,Female,Yes,Self-employed,Urban,never smoked,1,1


In [10]:
pycaret_predictions.columns.tolist()

['age',
 'hypertension',
 'heart_disease',
 'avg_glucose_level',
 'bmi',
 'gender',
 'ever_married',
 'work_type',
 'Residence_type',
 'smoking_status',
 'stroke',
 'prediction_label']

In [11]:
if "prediction_label" in pycaret_predictions.columns:
    pred_col = "prediction_label"
elif "Label" in pycaret_predictions.columns:
    pred_col = "Label"
else:
    raise ValueError("Could not find PyCaret prediction label column.")

y_test = pycaret_test_df["stroke"].to_numpy()
y_pred = pycaret_predictions[pred_col].astype(int).to_numpy()

if hasattr(best_model, "predict_proba"):
    X_test = pycaret_test_df.drop(columns=["stroke"])
    y_score = best_model.predict_proba(X_test)[:, 1]
else:
    y_score = None

np.unique(y_pred, return_counts=True), np.unique(y_test, return_counts=True)

((array([0, 1]), array([517, 250], dtype=int64)),
 (array([0, 1], dtype=int64), array([729,  38], dtype=int64)))

In [12]:
test_metrics = evaluate_binary_classifier(
    y_true=y_test,
    y_pred=y_pred,
    y_score=y_score,
)

test_metrics

{'accuracy': 0.7053455019556715,
 'precision': 0.124,
 'recall': 0.8157894736842105,
 'f1': 0.2152777777777778}

In [13]:
target_names = ["no_stroke", "stroke"]

report = create_classification_report_dict(
    y_true=y_test,
    y_pred=y_pred,
    target_names=target_names,
)

report

{'no_stroke': {'precision': 0.9864603481624759,
  'recall': 0.6995884773662552,
  'f1-score': 0.8186195826645265,
  'support': 729.0},
 'stroke': {'precision': 0.124,
  'recall': 0.8157894736842105,
  'f1-score': 0.2152777777777778,
  'support': 38.0},
 'accuracy': 0.7053455019556715,
 'macro avg': {'precision': 0.555230174081238,
  'recall': 0.7576889755252328,
  'f1-score': 0.5169486802211521,
  'support': 767.0},
 'weighted avg': {'precision': 0.9437308915390417,
  'recall': 0.7053455019556715,
  'f1-score': 0.7887278113663564,
  'support': 767.0}}

In [14]:
save_confusion_matrix_plot(
    y_true=y_test,
    y_pred=y_pred,
    class_names=target_names,
    output_path=METADATA_TRIAL_002_CONFUSION_MATRIX_PATH,
)

METADATA_TRIAL_002_CONFUSION_MATRIX_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/figures/experiments/metadata_trial_002_pycaret_confusion_matrix.png')

In [15]:
METADATA_TRIAL_002_PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

predictions_df = test_df.reset_index(drop=True).copy()
predictions_df["stroke_probability"] = y_score
predictions_df["predicted_label"] = y_pred
predictions_df["true_label"] = y_test
predictions_df["is_correct"] = predictions_df["predicted_label"] == predictions_df["true_label"]

predictions_df.to_csv(METADATA_TRIAL_002_PREDICTIONS_PATH, index=False)

METADATA_TRIAL_002_PREDICTIONS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/metadata_trial_002_pycaret_predictions.csv')

In [16]:
save_markdown_results(
    metrics=test_metrics,
    report=report,
    output_path=METADATA_TRIAL_002_RESULTS_PATH,
    title="Metadata Trial 002 PyCaret Benchmark Results",
)

METADATA_TRIAL_002_RESULTS_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/reports/experiments/metadata_trial_002_pycaret_benchmark_results.md')

In [17]:
stroke_recall = report["stroke"]["recall"]
stroke_f1 = report["stroke"]["f1-score"]
roc_auc = test_metrics.get("roc_auc", 0)

accepted = (
    stroke_recall >= 0.60
    and roc_auc >= 0.70
)

{
    "stroke_recall": stroke_recall,
    "stroke_f1": stroke_f1,
    "roc_auc": roc_auc,
    "accepted": accepted,
}

{'stroke_recall': 0.8157894736842105,
 'stroke_f1': 0.2152777777777778,
 'roc_auc': 0,
 'accepted': False}

In [18]:
if accepted:
    METADATA_TRIAL_002_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_model(best_model, str(METADATA_TRIAL_002_MODEL_PATH))
    print("Accepted PyCaret metadata model saved to:", METADATA_TRIAL_002_MODEL_PATH)
else:
    print("PyCaret model rejected. Results saved; model not saved.")

PyCaret model rejected. Results saved; model not saved.
